# 24 - Final Official RAG UI Demo

Clean Colab UI demo for the selected final Turkish Legal RAG system. Run the install cell, restart runtime once, then run the remaining cells.


In [ ]:
!python -m pip uninstall -y gradio gradio_client huggingface_hub transformers
!python -m pip install -q "gradio==4.44.1" "gradio_client==1.3.0" "huggingface_hub==0.34.4" "transformers==4.51.3" "sentence-transformers>=3.0.0" accelerate bitsandbytes peft faiss-cpu rank-bm25 pandas tqdm
print('Install finished. Now restart runtime once, then run the notebook from cell 2 onward.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {DRIVE_ROOT}')
os.chdir(DRIVE_ROOT)
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OFFICIAL_INDEX = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
BASE_LLM = 'Qwen/Qwen3-32B'
BASE_RERANKER = 'Qwen/Qwen3-Reranker-8B'
FINETUNED_LLM_ADAPTER = DRIVE_ROOT / 'models/adapters/qwen3_32b_qlora_combined_v1'

print('Working directory:', Path.cwd())
print('Device:', device)
print('Official index:', OFFICIAL_INDEX)
print('Official index manifest exists:', (OFFICIAL_INDEX / 'index_manifest.json').exists())
print('Fine-tuned LLM adapter exists:', FINETUNED_LLM_ADAPTER.exists())


In [ ]:
from functools import lru_cache
import gc
from pathlib import Path
from typing import Any

import torch
import gradio as gr

from src.generation import generate_text, load_llm
from src.prompting import build_rag_prompt
from src.reranking import Qwen3CausalRerankerDemo
from src.retrieval import RetrievalEngine

@lru_cache(maxsize=1)
def get_engine():
    return RetrievalEngine(index_root=OFFICIAL_INDEX, device=device)

def unload_cuda():
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def get_reranker(model_name: str):
    return Qwen3CausalRerankerDemo(model_name=model_name, device=device)

@lru_cache(maxsize=1)
def get_llm(model_name: str, adapter_path_text: str, load_in_4bit: bool):
    adapter_path = Path(adapter_path_text) if adapter_path_text else None
    return load_llm(
        model_name=model_name,
        device=device,
        load_in_4bit=load_in_4bit,
        adapter_path=adapter_path if adapter_path and adapter_path.exists() else None,
    )

def format_sources(items: list[dict[str, Any]]) -> str:
    lines = []
    for i, item in enumerate(items, start=1):
        citation = item.get('citation_label') or item.get('article_key') or ''
        article_key = item.get('article_key', '')
        score = item.get('score', '')
        text = str(item.get('generation_text') or item.get('retrieval_text') or '').replace('\n', ' ').strip()
        lines.append(f'[{i}] {citation}\narticle_key={article_key} | score={score}\n{text}')
    return '\n\n'.join(lines)

def retrieve_context(question: str, use_reranker: bool, candidate_k: int, top_k_context: int, reranker_batch_size: int):
    # Do not keep the embedding model cached while loading the 8B reranker.
    engine = RetrievalEngine(index_root=OFFICIAL_INDEX, device=device)
    if not use_reranker:
        try:
            return engine.dense_search(question, top_k=top_k_context)
        finally:
            del engine
            unload_cuda()

    candidates = engine.dense_search(question, top_k=candidate_k)
    del engine
    unload_cuda()

    reranker = None
    try:
        reranker = get_reranker(BASE_RERANKER)
        return reranker.rerank(
            query=question,
            candidates=candidates,
            text_field='retrieval_text',
            top_k=top_k_context,
            batch_size=reranker_batch_size,
        )
    finally:
        del reranker
        unload_cuda()

def answer_question(
    question: str,
    generate_answer: bool,
    use_reranker: bool,
    llm_variant: str,
    candidate_k: int,
    top_k_context: int,
    max_context_chars: int,
    max_new_tokens: int,
    reranker_batch_size: int,
):
    question = (question or '').strip()
    if not question:
        return 'Soru yaz.', ''
    if not (OFFICIAL_INDEX / 'index_manifest.json').exists():
        return f'Index bulunamad?: {OFFICIAL_INDEX}', ''

    try:
        retrieved = retrieve_context(
            question=question,
            use_reranker=use_reranker,
            candidate_k=int(candidate_k),
            top_k_context=int(top_k_context),
            reranker_batch_size=int(reranker_batch_size),
        )
    except Exception as exc:
        unload_cuda()
        return f'Retrieval/reranker hatas?: {type(exc).__name__}: {exc}', ''

    sources = format_sources(retrieved)
    if not generate_answer:
        mode = 'Qwen3-Embedding-8B + Qwen3-Reranker-8B' if use_reranker else 'Qwen3-Embedding-8B dense'
        return f'Retrieval-only mode: {mode}\n\n{sources}', sources

    unload_cuda()
    adapter_text = str(FINETUNED_LLM_ADAPTER) if llm_variant == 'Qwen3-32B QLoRA fine-tuned' else ''
    try:
        tokenizer, model = get_llm(BASE_LLM, adapter_text, True)
        prompt = build_rag_prompt(question, retrieved, max_context_chars=int(max_context_chars))
        answer = generate_text(
            tokenizer=tokenizer,
            model=model,
            prompt=prompt,
            max_new_tokens=int(max_new_tokens),
            temperature=0.0,
            top_p=1.0,
            input_max_length=8192,
        )
        return answer, sources
    except Exception as exc:
        get_llm.cache_clear()
        unload_cuda()
        return f'LLM hatas?. VRAM dolu olabilir; runtime restart edip tekrar deneyin. Hata: {type(exc).__name__}: {exc}', sources

with gr.Blocks(title='Turkish Legal RAG - Final System') as demo:
    gr.Markdown('## Turkish Legal RAG - Final Official System')
    gr.Markdown('Selected setup: Qwen3-Embedding-8B top-30 + Qwen3-Reranker-8B top-10. For a fast first test, leave LLM answer generation off.')

    question = gr.Textbox(label='Soru', lines=4, placeholder='Örn. Anayasa Mahkemesine bireysel başvuru şartları nelerdir?')

    with gr.Row():
        generate_answer = gr.Checkbox(value=False, label='LLM cevabı üret')
        use_reranker = gr.Checkbox(value=True, label='Qwen3-Reranker-8B kullan')
        llm_variant = gr.Dropdown(
            choices=['Qwen3-32B base', 'Qwen3-32B QLoRA fine-tuned'],
            value='Qwen3-32B base',
            label='LLM varyant?',
        )

    with gr.Row():
        candidate_k = gr.Slider(10, 50, value=30, step=5, label='Candidate K')
        top_k_context = gr.Slider(3, 15, value=10, step=1, label='Context Top K')
        reranker_batch_size = gr.Slider(1, 8, value=4, step=1, label='Reranker batch size')

    with gr.Row():
        max_context_chars = gr.Slider(3000, 16000, value=9000, step=1000, label='Max context chars')
        max_new_tokens = gr.Slider(128, 768, value=384, step=64, label='Max answer tokens')

    run_btn = gr.Button('Çalıştır')
    answer_box = gr.Textbox(label='Cevap / Retrieval sonucu', lines=16)
    sources_box = gr.Textbox(label='Kaynaklar', lines=16)

    run_btn.click(
        answer_question,
        inputs=[question, generate_answer, use_reranker, llm_variant, candidate_k, top_k_context, max_context_chars, max_new_tokens, reranker_batch_size],
        outputs=[answer_box, sources_box],
    )

demo.launch(share=True, debug=True, inline=False)
